# Gameweek planner

Expected points for **every** player in the game, plus your own squad analysed.

Set your team in section 1, then run top to bottom. Everything except sections
4–6 works without a team set.

## 1. Settings - edit these

In [ ]:
# ─── YOUR TEAM ────────────────────────────────────────────────────────────
# Leave as None to read FPL_TEAM_ID from .env (set it once and forget it).
# Set a number here to look at a different team just for this run.
#
# To find an id: view the team on the FPL site and read it out of the URL -
#     fantasy.premierleague.com/entry/ 1234567 /event/4
#                                      ^^^^^^^
TEAM_ID = None

# Or list your 15 by name instead (used only if TEAM_ID is None):
MY_TEAM = []              # e.g. ["Haaland", "Saka", "Guéhi", ...]

# ─── MINUTES ──────────────────────────────────────────────────────────────
# Playing time is modelled automatically (availability flags, recent starts,
# rotation risk, and the rule that each team starts exactly 11). Where you know
# better than the model, override it per player in
# config/minutes_overrides.yaml and re-run:
#
#     players:
#       Haaland: 90     # pin to a full match
#       Saka: 0         # rule out
#
# Section 10 lists the players most worth a manual call.

# ─── FIXTURE OUTLOOK (section 7) ──────────────────────────────────────────
LOOKAHEAD = 10            # how many gameweeks to chart
WINDOW    = 5             # how many to rank teams over
FREE_TRANSFERS = 1        # how many transfers you have without taking a hit
DISCOUNT  = 0.85          # weight on gameweek k = DISCOUNT ** (k-1)
                          #   1.0  = every gameweek in the window counts equally
                          #   0.85 = balanced (default)
                          #   0.7  = concentrate on the next two or three

# ─── OTHER ────────────────────────────────────────────────────────────────
GAMEWEEK = None           # None = next gameweek

## 2. Run the model

In [ ]:
# Pick up edits to the fplfh package without restarting the kernel. Without
# this, a running kernel keeps the version of a module it first imported, and
# any function added since fails to import until you restart.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass          # not running under IPython

import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "fplfh").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

from fplfh.pipeline import run
from fplfh.fpl import FPLClient
from fplfh.optimise import best_xi, optimise_free_hit
from fplfh.model import COMPONENTS

client = FPLClient()
res = run(event=GAMEWEEK)

print(f"\nGameweek {res.event}   |   "
      f"{100*res.exchange_share():.0f}% of expected points from market odds")
res.fixture_table()

## 3. Every player, ranked

The full table. `xp` is the sum of the ten component columns - reading the
components tells you *why* a player rates, which is usually more useful than the
total.

In [ ]:
ALL = res.players.copy()
cols = ["web_name","team","position","price","opponent","is_home"] + COMPONENTS + ["xp"]

print(f"{len(ALL)} players projected for GW{res.event}\n")
ALL.head(40)[cols].round(2)

In [ ]:
# Filter it however you like - a few useful starting points.
def top(position=None, max_price=None, min_xp=0.0, team=None, n=20):
    d = ALL.copy()
    if position:  d = d[d.position == position]
    if max_price: d = d[d.price <= max_price]
    if team:      d = d[d.team == team]
    d = d[d.xp >= min_xp]
    return d.head(n)[["web_name","team","position","price","opponent","xmins","xp"]].round(2)

print("Best value under £6.0m:")
display(top(max_price=6.0, n=12))

In [ ]:
print("Top 8 in each position:\n")
for pos in ("GKP","DEF","MID","FWD"):
    print(f"── {pos} " + "─"*60)
    print(top(position=pos, n=8).to_string(index=False))
    print()

In [ ]:
# Points per million - for filling out the cheap end of a squad
v = ALL[ALL.xp > 1.0].copy()
v["xp_per_million"] = v.xp / v.price
print("Best value per million (xP > 1.0):")
v.nlargest(20, "xp_per_million")[
    ["web_name","team","position","price","opponent","xp","xp_per_million"]].round(2)

## 4. Your team

Pulled straight from the FPL API using `FPL_TEAM_ID` from `.env`, or `TEAM_ID`
if you set one above.

Note the API only makes picks public once a gameweek's deadline has passed, so
this shows your most recent **confirmed** squad - transfers you have made since
will not appear.

In [ ]:
from fplfh.config import fpl_team_id

# An explicit TEAM_ID above wins; otherwise fall back to .env.
team_id = TEAM_ID if TEAM_ID else fpl_team_id()
squad_df, my_budget, source = None, None, None

if team_id:
    picks, picked_gw = client.latest_picks(int(team_id))
    ids = [p["element"] for p in picks["picks"]]
    squad_df = ALL[ALL.player_id.isin(ids)].copy()
    eh = picks["entry_history"]
    my_budget = (eh["value"] + eh["bank"]) / 10.0
    info = client.entry(int(team_id))
    source = f'{info.get("name","?")} ({info.get("player_first_name","")} '\
             f'{info.get("player_last_name","")}) - squad as at GW{picked_gw}'
    print(source)
    print(f"  squad value L{eh['value']/10:.1f}m + bank L{eh['bank']/10:.1f}m "
          f"= L{my_budget:.1f}m")
    print(f"  overall rank {info.get('summary_overall_rank'):,}" if info.get("summary_overall_rank") else "")
    missing = set(ids) - set(squad_df.player_id)
    if missing:
        print(f"  note: {len(missing)} player(s) have no fixture this gameweek (blank)")

elif MY_TEAM:
    # Accent-insensitive on both sides: nobody types "Ødegaard" or "Groß" with
    # the special letters, and those are atomic in Unicode so a naive accent
    # strip mangles them ("Hojlund" -> "h jlund"). normalise() transliterates.
    from fplfh.naming import normalise

    ALL["_key"] = ALL.web_name.map(normalise)
    # normalise() strips brackets, so "Palmer (CHE)" becomes "palmer che" -
    # the key has to be built the same way or the qualifier never matches.
    ALL["_key_team"] = ALL._key + " " + ALL.team.str.lower()

    # 19 surnames are shared by two or three players (three Wilsons, three
    # Phillips, two Palmers), so a bare surname is genuinely ambiguous. Qualify
    # those with a club: "Palmer (CHE)".
    counts = ALL._key.value_counts()
    picked, unmatched, ambiguous = [], [], []
    for raw in MY_TEAM:
        k = normalise(raw)
        exact = ALL[ALL._key_team == k]
        if len(exact) == 1:
            picked.append(exact.index[0]); continue
        hits = ALL[ALL._key == k]
        if len(hits) == 1:
            picked.append(hits.index[0])
        elif len(hits) > 1:
            ambiguous.append((raw, hits[["web_name", "team", "position", "xp"]]))
        else:
            unmatched.append(raw)

    squad_df = ALL.loc[picked].copy()

    for raw, hits in ambiguous:
        print(f"  AMBIGUOUS '{raw}' - {len(hits)} players share that name. "
              f'Use e.g. "{raw} ({hits.iloc[0].team})":')
        print(hits.to_string(index=False))
    for raw in unmatched:
        close = ALL[ALL._key.str.startswith(normalise(raw)[:4])].web_name.head(4).tolist()
        print(f"  NOT FOUND '{raw}'" + (f" - did you mean {close}?" if close else ""))

    ALL = ALL.drop(columns=["_key", "_key_team"])
    squad_df = squad_df.drop(columns=["_key", "_key_team"])
    my_budget = squad_df.price.sum()
    source = "manual list"
    print(f"matched {len(squad_df)} of {len(MY_TEAM)} names, "
          f"total value L{my_budget:.1f}m")

else:
    print("No team set. Either put FPL_TEAM_ID in .env, or fill in TEAM_ID or")
    print("MY_TEAM in section 1, to use sections 4-6.")
    print("Everything else in this notebook works without it.")

In [ ]:
if squad_df is not None and len(squad_df):
    d = squad_df.sort_values(["position","xp"], ascending=[True,False])
    print(f"Your {len(d)} players, projected for GW{res.event}:\n")
    display(d[["web_name","team","position","price","opponent","is_home","xmins"]
              + COMPONENTS + ["xp"]].round(2))
    print(f"\nsquad total xP (all {len(d)}): {d.xp.sum():.2f}")

### Your best XI

Which of your 15 to start, who to captain, and the bench order. This chooses
nothing about *which* players you own - only how to line them up.

In [ ]:
if squad_df is not None and len(squad_df) >= 11:
    mine = best_xi(squad_df)
    print(mine.summary())
    print(f"\nprojected total: {mine.starting_xp:.2f} points (captain doubled)")
    bench_pts = squad_df[~squad_df.player_id.isin(
        mine.players[mine.players.is_starter].player_id)].xp.sum()
    print(f"left on the bench: {bench_pts:.2f}")

In [ ]:
# Captain choice - the armband is worth a whole extra return, so it is the
# single biggest decision of the week.
if squad_df is not None and len(squad_df):
    print("Captaincy options from your squad:\n")
    display(squad_df.nlargest(5, "xp")[
        ["web_name","team","position","opponent","is_home","xp_goals","xp_assists",
         "xmins","xp"]].round(2))
    print("Doubling the top pick is worth an extra "
          f"{squad_df.xp.max():.2f} points in expectation.")

## 5. Transfers

For each player you own, the best alternative in the same position you could
afford. Selling price is approximated as the current price - FPL's actual rule
gives back half of any profit, so real funds may be slightly lower.

In [ ]:
if squad_df is not None and len(squad_df):
    bank = 0.0 if team_id is None else (client.latest_picks(int(team_id))[0]
                                       ["entry_history"]["bank"] / 10.0)
    owned = set(squad_df.player_id)
    rows = []
    for _, p in squad_df.iterrows():
        budget = p.price + bank
        cands = ALL[(ALL.position == p.position) & (~ALL.player_id.isin(owned))
                    & (ALL.price <= budget) & (ALL.xp > p.xp)]
        if not len(cands):
            continue
        b = cands.nlargest(1, "xp").iloc[0]
        rows.append({"out": p.web_name, "out_team": p.team, "out_xp": round(p.xp,2),
                     "in": b.web_name, "in_team": b.team, "in_xp": round(b.xp,2),
                     "gain": round(b.xp - p.xp,2),
                     "cost": round(b.price - p.price,1)})
    if rows:
        t = pd.DataFrame(rows).sort_values("gain", ascending=False)
        print(f"Best single upgrade per player (bank L{bank:.1f}m):\n")
        display(t)
        print("A transfer costs 4 points unless it is free - only the top few")
        print("rows are likely to be worth taking a hit for.")
    else:
        print("No affordable upgrades found - your squad is already strong "
              "for this gameweek.")

## 6. The unconstrained best squad

What the optimiser would pick if you could start from scratch with your budget.
Useful as a benchmark - the gap between this and your XI is what a Free Hit or
Wildcard could theoretically buy you.

In [ ]:
# A partial or empty squad gives a budget far too small for 15 players, so fall
# back to the game's standard 100.0 rather than asking for the impossible.
budget = my_budget if (my_budget and my_budget >= 80) else 100.0
if my_budget and my_budget < 80:
    print(f"(matched squad is only worth L{my_budget:.1f}m, too little for a "
          f"full 15 - using the standard L100.0m instead)\n")

dream = optimise_free_hit(ALL, budget=budget)
if dream is None:
    print(f"No legal 15-player squad fits inside L{budget:.1f}m.")
else:
    print(f"Best possible squad for L{budget:.1f}m:\n")
    print(dream.summary())

if dream is not None and squad_df is not None and len(squad_df) >= 11:
    print(f"\n{'─'*64}")
    print(f"  your XI          {mine.starting_xp:6.2f}")
    print(f"  best possible    {dream.starting_xp:6.2f}")
    print(f"  gap              {dream.starting_xp - mine.starting_xp:6.2f} points")
    print("\n  That gap needs 15 transfers to close, so treat it as a ceiling,")
    print("  not a target.")

## 7. Fixture outlook

Who has the best run of games, and over what horizon.

Difficulty here isn't FPL's 1-5 rating, which is a coarse integer set before
the season and never revised. It comes from the same scoreline model as
everything else, in the units that actually score points: expected goals for
(attackers), clean sheet probability (defenders), expected goals against
(keepers).

The odds themselves only reach three to four gameweeks ahead - on the live
feed that was 24 days, covering two gameweeks across an international break.
Beyond that the ratings model fills in, and every fixture is tagged with which
was used.

Distant gameweeks are weighted down in the ranking, but not because they're
less predictable: measured across a full season, the correlation between a
team's attacking rating and its actual goals is flat at ~0.28-0.31 whether you
look one gameweek ahead or ten, so team strength is broadly stable over that
range. The discount is really about actionability - you'll make transfers
before then, so a far-off fixture should sway today's decision less. That
makes it a planning preference, not a fitted constant.

In [ ]:
from fplfh.outlook import build_outlook, fixture_grid, rank_fixtures, compare_windows

outlook = build_outlook(client, res.minutes, res.fixtures, res.event,
                        n_events=LOOKAHEAD, verbose=True)
n_mkt = int((outlook.source == "market").sum())
print()
print(f"{len(outlook)} team-fixtures over GW{res.event}-{res.event + LOOKAHEAD - 1}")
print(f"  {n_mkt} priced from the market, {len(outlook) - n_mkt} from the ratings model")
print()
print("The schedule (H = home, A = away):")
fixture_grid(outlook)

In [ ]:
# The same grid as numbers - expected goals FOR each team, per gameweek.
# Blanks show as NaN; doubles are summed.
print("Expected goals for, by gameweek:")
fixture_grid(outlook, value="xg_for").round(2)

### Best fixtures over your chosen window

In [ ]:
for metric, who in (("attack", "forwards and attacking midfielders"),
                    ("defence", "defenders and goalkeepers")):
    r = rank_fixtures(outlook, window=WINDOW, discount=DISCOUNT, metric=metric)
    print(f"-- BEST for {metric.upper()} over the next {WINDOW} GWs ({who})")
    print(r.head(6)[["fixtures", "blanks", "doubles", "xg_for", "clean_sheet",
                     "home_games", "market_priced", "opponents"]].round(3).to_string())
    print(f"   avoid: {', '.join(r.tail(3).index)}")
    print()

In [ ]:
# Does the discount change the answer? If a team's rank holds across discounts
# its run is uniformly good; if it swings, the good fixtures are clustered at
# one end of the window.
# Ranks each team at several discounts, to show how sensitive the order is.
# Always includes whatever DISCOUNT is set above, so the sort column exists
# whichever value you choose.
def discount_swing(metric, discounts=(1.0, 0.85, 0.7, 0.5)):
    ranks = {}
    for d in sorted(set(discounts) | {float(DISCOUNT)}, reverse=True):
        label = f"discount {d:g}" + ("  <-yours" if d == float(DISCOUNT) else "")
        ranks[label] = rank_fixtures(
            outlook, window=WINDOW, discount=d, metric=metric
        ).score.rank(ascending=False).astype(int)
    t = pd.DataFrame(ranks)
    t["swing"] = t.max(axis=1) - t.min(axis=1)
    return t.sort_values(next(c for c in t.columns if "<-yours" in c))


print("ATTACK - rank by discount. A large swing means the good fixtures are")
print("clustered early or late rather than spread evenly:")
display(discount_swing("attack").head(12))

In [ ]:
print("DEFENCE - rank by discount.")
print("Clean-sheet probability varies proportionally more across teams than")
print("expected goals does (coefficient of variation ~0.22 vs ~0.14), so WHICH")
print("fixture a defender has matters relatively more. The rank swing itself is")
print("similar for both, though - neither is reliably the more volatile.")
display(discount_swing("defence").head(12))

In [ ]:
# Short run vs long run - who to buy now, and who to wait for.
print("ATTACK - rank over different windows:")
display(compare_windows(outlook, windows=(3, WINDOW, LOOKAHEAD), metric="attack").head(12))
print("A team that improves as the window lengthens has a hard patch first -")
print("worth planning for rather than buying today.")

In [ ]:
print("DEFENCE - rank over different windows:")
display(compare_windows(outlook, windows=(3, WINDOW, LOOKAHEAD), metric="defence").head(12))
print("Compare with the attack table above: a team can be a good short-term")
print("defensive pick and a poor attacking one, or the reverse.")

In [ ]:
# Your own players' fixture runs
if squad_df is not None and len(squad_df):
    rk = rank_fixtures(outlook, window=WINDOW, discount=DISCOUNT, metric="overall")
    mine_fx = (squad_df[["web_name", "team", "position", "price", "xp"]]
               .merge(rk[["score", "xg_for", "clean_sheet", "blanks", "opponents"]],
                      left_on="team", right_index=True, how="left")
               .sort_values("score", ascending=False))
    print(f"Your squad by fixture run over the next {WINDOW} gameweeks:")
    display(mine_fx.round(3))
    print("Players at the bottom are the natural transfer candidates, even if")
    print("their expected points this week look fine.")

## 8. Multi-gameweek player ranking

Section 7 ranks *teams* by fixtures. This ranks *players* over the same
window, which is what a transfer decision actually needs - fixtures matter,
but so do minutes, role and price.

Each player's expected points are summed across the window with the same
`DISCOUNT` weighting.

### The assumptions, stated plainly

Minutes are frozen at today's estimate, except where FPL gives a return date.
Only about 13% of flagged players carry one ("Expected back 11 Oct"); those
switch back on at the right gameweek. The rest hold today's availability
across the whole window - deliberately, since we don't know when they return,
and a made-up recovery curve would look like information while being a guess.

A returning player's start rate comes from the price/position prior rather
than their own record, because their record is a run of zeros *caused by* the
injury and says nothing about whether they'd be picked when fit - several
have played no minutes at all this season. Those rows are tagged
`minutes_basis = "prior"` and flagged below; treat them as the roughest
numbers in the table.

Market odds only reach two to four gameweeks, and `market_share` reports how
much of each player's total came from priced fixtures rather than the ratings
model.

Prices are also frozen, even though over a long window they move and that
affects what you can afford later. Not modelled here.

In [ ]:
from fplfh.availability import return_gameweeks
from fplfh.horizon import (player_horizon, gameweek_matrix,
                           suggest_transfers, transfer_summary)

return_gw = return_gameweeks(res.minutes, client.bootstrap()["events"])
print(f"{len(return_gw)} flagged players have a stated return gameweek")

HZN, per_gw = player_horizon(client, res.minutes, res.fixtures, res.event,
                             n_events=WINDOW, discount=DISCOUNT,
                             return_gw=return_gw, verbose=True)
print()
print(f"ranked {len(HZN)} players over GW{res.event}-{res.event + WINDOW - 1}, "
      f"discount {DISCOUNT}")
HZN.head(25)[["web_name", "team", "position", "price", "xp_total", "xp_raw",
              "xp_next", "fixtures", "blanks", "market_share",
              "xp_per_million"]].round(2)

In [ ]:
# Best value per million over the window, rather than for one gameweek
print("Best value per million over the window (xp_total > 5):")
display(HZN[HZN.xp_total > 5].nlargest(15, "xp_per_million")[
    ["web_name", "team", "position", "price", "xp_total", "xp_per_million",
     "blanks"]].round(2))

In [ ]:
# Who is riding a prior rather than a record? These are the shakiest rows.
shaky = HZN[HZN.get("minutes_basis", "observed") != "observed"]
if len(shaky):
    print("Returning from injury - minutes rest on the price/position prior,")
    print("not on anything observed. Sanity-check these by eye:")
    display(shaky.nlargest(10, "xp_total")[
        ["web_name", "team", "position", "price", "xp_total", "xmins_mean"]].round(2))
else:
    print("No players in the window are running on a prior.")

In [ ]:
# The shape of a run matters as much as the total - a flat 5 a week is worth
# more to plan around than a 12 followed by four blanks.
print("Expected points by gameweek, top 8 over the window:")
gameweek_matrix(per_gw, HZN.head(8).web_name.tolist()).round(2)

## 9. Transfers

For each player you own, the best affordable replacement over the window.

This is a deliberately simple greedy view: it takes the single biggest
upgrade, strikes both players off, then finds the next best among what's
left - tracking the bank and the three-per-club limit as it goes, so the list
is actually executable in order rather than just a set of individually
plausible swaps.

It doesn't consider combinations (selling two cheap players to fund one
expensive one), plan across future gameweeks, or model free-transfer banking.
Those would be a much larger optimisation, worth building only once the
projections have been validated enough to trust at that resolution.

Read `net` carefully: the gain is a whole-window total, while the −4 hit is
paid once. And a net under about a point is well inside the model's own
error - the backtest puts rank correlation around 0.69, useful for ordering
players but not for splitting hairs.

In [ ]:
if squad_df is not None and len(squad_df):
    my_ids = squad_df.player_id.tolist()
    my_bank = 0.0
    if team_id:
        my_bank = client.latest_picks(int(team_id))[0]["entry_history"]["bank"] / 10.0

    owned = HZN[HZN.player_id.isin(my_ids)].sort_values("xp_total", ascending=False)
    print(f"Your squad over GW{res.event}-{res.event + WINDOW - 1} "
          f"(bank L{my_bank:.1f}m):")
    display(owned[["web_name", "team", "position", "price", "xp_total",
                   "xp_next", "blanks", "market_share"]].round(2))
    print(f"squad total (discounted): {owned.xp_total.sum():.1f}")
else:
    print("No team set - see section 1.")

In [ ]:
if squad_df is not None and len(squad_df):
    sug = suggest_transfers(HZN, my_ids, bank=my_bank)
    plan = transfer_summary(sug, bank=my_bank, squad_teams=owned.team.tolist(),
                            free_transfers=FREE_TRANSFERS)
    if len(plan):
        print(f"{len(sug)} legal upgrades found. Best executable sequence:")
        display(plan.head(10).round(2))
        print("bank_after tracks the money left once each swap is done, so the")
        print("sequence is affordable in the order shown.")
        good = plan[plan.net > 1.0]
        print()
        print(f"{len(good)} of these clear a net of +1.0, which is roughly the")
        print("threshold below which the model cannot tell the difference.")
    else:
        print("No affordable upgrades found - your squad is already strong for")
        print("this window, or the bank is too thin to move.")

## 10. Minutes worth checking by hand

The model reads FPL's availability flags but cannot hear a press conference.
These are the players where its guess matters most and is least certain - worth
overriding in `config/minutes_overrides.yaml` if you know better, then re-running.

In [ ]:
m = res.minutes
risky = res.players.merge(
    m[["player_id","p_start","p_60","status","news"]], on="player_id", suffixes=("","_m"))
risky = risky[(risky.xp > 1.5) & (risky.p_start.between(0.2, 0.85))]
print("Rotation risks among players worth owning:\n")
display(risky.nlargest(15, "xp")[
    ["web_name","team","position","price","opponent","p_start","p_60","xmins","xp"]].round(3))

flagged = m[(m.status != "a") & (m.price >= 4.5)]
if len(flagged):
    print("\nFlagged players (automatically downgraded):\n")
    display(flagged.nlargest(12, "price")[
        ["web_name","team","position","price","status","chance_of_playing",
         "p_start","xmins","news"]])

## 11. Export

In [ ]:
from fplfh.config import OUT_DIR, ensure_dirs
ensure_dirs()
tag = f"gw{res.event}"
ALL.to_csv(OUT_DIR / f"all_players_{tag}.csv", index=False)
res.fixture_table().to_csv(OUT_DIR / f"fixtures_{tag}.csv", index=False)
if squad_df is not None and len(squad_df):
    squad_df.to_csv(OUT_DIR / f"my_squad_{tag}.csv", index=False)
print("written to", OUT_DIR)